In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import xarray as xr
import numpy as np
import os
from glob import glob
from mpl_toolkits.basemap import Basemap
from numpy import meshgrid
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib import colormaps
import pandas as pd
import math
from datetime import datetime
import datetime as dt
from ridgeplot import ridgeplot
import joypy
import seaborn as sns
from matplotlib import cm
import climpred
from xclim import sdba
from climpred.options import OPTIONS
import json
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from matplotlib.lines import Line2D  # For custom legend entries
import warnings
from sklearn.exceptions import UndefinedMetricWarning
import matplotlib.gridspec as gridspec
import hydroeval as he
import re
from matplotlib.colors import Normalize
import matplotlib.colors as mcolors

from function import preprocessUtils as putils
from function import masks
from function import verifications
from function import funs as f
from function import conf
from function import loadbias
from function import quikplot as qp
from function import dataLoad


warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

global dim_order, region_name, test_year, leads_
dim_order = conf.dim_order

test_year = 2019
leads_ = [6,13,20,27]

dir = '/glade/work/klesinger/FD_RZSM_deep_learning'
assert test_year == 2019, 'This is only the script for when the testing years are 2018-2019. Test year must = 2019.'


## This script only works if there is a single experiment done for china and australia
### We are doing EX29 only
### For both ECMWF and GEFSv12
## For both ERA5-land and GLEAM

In [ ]:
region_name = 'china' #['australia','china', 'CONUS']
obs_source = 'GLEAM' #['GLEAM','ERA5']

if obs_source == 'ERA5':
    soil_dir = conf.era_data
elif obs_source == 'GLEAM':
    soil_dir = conf.gleam_data

In [ ]:
mask, mask_anom = masks.load_mask_vals(region_name)

In [ ]:
global custom_names
'''This is for the final plot for ACC and CRPS'''
custom_names = {
    'GEFSv12': 'GEFSv12','GEFSv12-BC': 'GEFSv12-BC', 'DL-DM_GEFSv12': 'DL-DM_G','DL-DM_ECMWF': 'DL-DM_E',
    'ECMWF':'ECMWF', 'ECMWF-BC':'ECMWF-BC',
}

        
def return_name(name):
    if 'XGBOOST' in name:
        name_out = 'ML_NWP_OBS'
    else:
        name_out = name
    custom_names = {name: name_out}

    return(custom_names)

In [ ]:
'''Testing and validation dates only for the year 2019'''

test_start = '2018-01-01'
test_end = '2019-12-31'
val_start = '2016-01-01'
val_end = '2017-12-31'
train_start  = '2000-01-01'


'''Test subsets of obs, ecmwf raw, gefsv12 raw '''
global obs_anomaly_SubX_format, baseline_gefs, baseline_ecmwf, var_OUT, template_testing_only
obs_anomaly_SubX_format, baseline_gefs, baseline_ecmwf, var_OUT, template_testing_only = verifications.open_obs_and_baseline_files_multiple_leads(region_name, leads_, test_start, test_end, mask_anom, soil_dir)

init_dates, dt_dates, only_testing_dates = dataLoad.return_init_and_testing_dates(region_name,test_start,test_end)


In [ ]:
global obs_original,obs_raw
obs_original,obs_raw = dataLoad.load_rzsm_observations(soil_dir, region_name)
obs_original["time"] = obs_original["time"].dt.floor("D")
obs_raw["time"] = obs_raw["time"].dt.floor("D")

# Lineplot ACC and CRPS all experiments

In [ ]:

def add_lineplot_to_dataframe(df,fcst_vals,name_of_fcst, metric,week, mean_or_median):
    # df = pd.DataFrame()
    def return_color(name_of_fcst):
        black = ['EX0','EX13'] # bias-corrected DL
        red = ['EX14','EX15','EX16','EX17','EX22','EX23','EX24','EX25'] #Observation driven
        blue = ['EX1','EX2','EX3','EX4','EX5','EX6','EX7','EX8','EX9','EX10','EX11','EX12',
               'EX18','EX19','EX20','EX21','EX27','EX28','EX29'] #Hybrid

        black2 = ['DM-BC_DL']
        red2 = ['DL']
        blue2 = ['DL-DM']
        
        green = ['ECMWF','GEFSv12']
        
        purple = 'GEFSv12'

        orange = 'ECMWF'

        yellow = 'EMOS'

        if (name_of_fcst in black) or (name_of_fcst in black2):
            color = 'black'
        elif (name_of_fcst in red) or (name_of_fcst in red2):
            color = 'red'
        elif (name_of_fcst in blue) or (name_of_fcst in blue2):
            color = 'blue'
        elif name_of_fcst in green:
            color = 'green'
        elif purple in name_of_fcst:
            color = 'purple'
        elif yellow in name_of_fcst:
            color = 'yellow'
        elif orange in name_of_fcst:
            color='orange'
        return(color)

    if week==10:
        for idx,lead in enumerate([6,13,20,27]):
            if mean_or_median == 'mean':
                try:
                    data = fcst_vals.sel(lead=lead).mean()[putils.xarray_varname(fcst_vals)].values
                except KeyError:
                    data = fcst_vals.mean()[putils.xarray_varname(fcst_vals)].values
            if mean_or_median == 'median':
                try:
                    data = fcst_vals.sel(lead=lead).median()[putils.xarray_varname(fcst_vals)].values
                except AttributeError:
                    data = np.nanmedian(fcst_vals[idx,:,:])
            dict_ = {'Forecast':[name_of_fcst], 'Week':[idx+1], f'{metric}': [data], 'Color':return_color(name_of_fcst)}
            df = pd.concat([df,pd.DataFrame.from_dict(dict_)])
    else:
        if mean_or_median == 'mean':
            data = fcst_vals.mean()[putils.xarray_varname(fcst_vals)].values
        elif mean_or_median == 'median':
            data = fcst_vals.median()[putils.xarray_varname(fcst_vals)].values
        dict_ = {'Forecast':[name_of_fcst], 'Week':[week], f'{metric}': [data], 'Color':return_color(name_of_fcst)}
        df = pd.concat([df,pd.DataFrame.from_dict(dict_)])

    return(df)


'''Only the single value for all the experiments'''

def add_lineplot_to_dataframe_average(df,fcst_vals,name_of_fcst, metric,week, mean_or_median):
    # df = pd.DataFrame()

    def return_color(name_of_fcst):
        black = ['DM-BC_DL']
        red = ['DL']
        blue = ['DL-DM_GEFSv12']
        purple = ['DL-DM_ECMWF']
        
        green = ['ECMWF','GEFSv12']
        

        yellow = 'EMOS'

        if name_of_fcst in black:
            color = 'black'
        elif name_of_fcst in red:
            color = 'red'
        elif name_of_fcst in blue:
            color = 'blue'
        elif name_of_fcst in green:
            color = 'green'
        elif name_of_fcst in purple:
            color = 'purple'
        elif name_of_fcst in yellow:
            color = 'yellow'
            
        return(color)

    if week==10:
        for idx,lead in enumerate([6,13,20,27]):
            if mean_or_median == 'mean':
                data = fcst_vals.sel(lead=lead).mean()[putils.xarray_varname(fcst_vals)].values
            elif mean_or_median == 'median':
                try:
                    data = fcst_vals.sel(lead=lead).median()[putils.xarray_varname(fcst_vals)].values
                except AttributeError:
                    data = np.nanmedian(fcst_vals[idx,:,:])
            dict_ = {'Forecast':[name_of_fcst], 'Week':[idx+1], f'{metric}': [data], 'Color':return_color(name_of_fcst)}
            df = pd.concat([df,pd.DataFrame.from_dict(dict_)])
    else:
        if mean_or_median == 'mean':
            data = fcst_vals.mean()[putils.xarray_varname(fcst_vals)].values
        elif mean_or_median == 'median':
            data = fcst_vals.median()[putils.xarray_varname(fcst_vals)].values
            
        dict_ = {'Forecast':[name_of_fcst], 'Week':[week], f'{metric}': [data], 'Color':return_color(name_of_fcst)}
        df = pd.concat([df,pd.DataFrame.from_dict(dict_)])
    
    return(df)

def add_lineplot_to_dataframe_average_by_week(df,fcst_vals,name_of_fcst, metric,week, mean_or_median, lead, idLead):
    # df = pd.DataFrame()

    # df = df_crps_djf
    # fcst_vals = seasonal_crps[season]
    # name_of_fcst = name
    # metric = 'CRPS'
    # week = 10
    # mean_or_median = 'median'
    # lead=day_num
    # idLead = idLead
    
    def return_color(name_of_fcst):
        black = ['DM-BC_DL']
        red = ['DL']
        blue = ['DL-DM_GEFSv12']
        purple = ['DL-DM_ECMWF']
        
        green = ['ECMWF','GEFSv12']


        yellow = 'EMOS'

        if name_of_fcst in black:
            color = 'black'
        elif name_of_fcst in red:
            color = 'red'
        elif name_of_fcst in blue:
            color = 'blue'
        elif name_of_fcst in green:
            color = 'green'
        elif name_of_fcst in purple:
            color = 'purple'
        elif name_of_fcst in yellow:
            color = 'yellow'
            
        return(color)

    if week==10:
        if mean_or_median == 'mean':
            data = fcst_vals.sel(lead=lead).mean()[putils.xarray_varname(fcst_vals)].values
        elif mean_or_median == 'median':
            data = fcst_vals.sel(lead=lead).median()
        dict_ = {'Forecast':[name_of_fcst], 'Week':[idLead+1], f'{metric}': [data], 'Color':return_color(name_of_fcst)}
        try:
            df = pd.concat([df,pd.DataFrame.from_dict(dict_)])
        except TypeError:
            dict_ = {'Forecast':[name_of_fcst], 'Week':[idLead+1], f'{metric}': [data[putils.xarray_varname(data)].values], 'Color':return_color(name_of_fcst)}
            df = pd.concat([df,pd.DataFrame.from_dict(dict_)])
            
    else:
        if mean_or_median == 'mean':
            data = fcst_vals.mean()[putils.xarray_varname(fcst_vals)].values
        elif mean_or_median == 'median':
            data = fcst_vals.median()[putils.xarray_varname(fcst_vals)].values
            
        dict_ = {'Forecast':[name_of_fcst], 'Week':[week], f'{metric}': [data], 'Color':return_color(name_of_fcst)}
        df = pd.concat([df,pd.DataFrame.from_dict(dict_)])
    
    return(df)

In [ ]:
def common_UNET_experiments(correct_experiments, obs_source):
    only_RZSM = [j for j in correct_experiments if 'RZSM' in j] 
    only_ensemble= [j for j in only_RZSM if 'final' not in j]
    only_ensemble = [j for j in only_ensemble if 'Residual' not in j]
    only_2019 = [j for j in only_ensemble if '2012' not in j]
    only_2019 = [j for j in only_ensemble if '2019' not in j] #This is an old experiment
    if obs_source == 'GLEAM':
        only_2019 = [j for j in only_ensemble if 'ERA5' not in j]
    elif obs_source == 'ERA5':
        only_2019 = [j for j in only_ensemble if 'ERA5' in j]
    return(only_2019)

def common_UNET_no_regular_experiments(correct_experiments):
    only_RZSM = [j for j in correct_experiments if 'RZSM' in j] 
    only_ensemble= [j for j in only_RZSM if 'final' not in j]
    only_ensemble = [j for j in only_ensemble if 'Residual' not in j]
    only_ensemble = [j for j in only_ensemble if 'regular' not in j]
    only_2019 = [j for j in only_ensemble if '2012' not in j]
    return(only_2019)

In [ ]:


def filter_files_by_ex_GEFS(file_list, color_list, week_,obs_source):
    filtered_files = []
    
    for file in file_list:
        if obs_source== 'GLEAM':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_regular_RZSM', file)
        elif obs_source == 'ERA5':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_regular_ERA5_RZSM', file)
        if match:
            ex_value = f"EX{match.group(1)}"
            if ex_value in color_list:
                filtered_files.append(file)
    
    return filtered_files

def filter_files_by_ex_ECMWF(file_list, color_list, week_,obs_source):
    filtered_files = []
    
    for file in file_list:
        if obs_source =='GLEAM':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_ECMWF_regular_RZSM', file)
        elif obs_source == 'ERA5':
            match = re.search(rf'Wk{week_}_testing_EX(\d+)_ECMWF_regular_ERA5_RZSM', file)
        if match:
            ex_value = f"EX{match.group(1)}"
            if ex_value in color_list:
                filtered_files.append(file)
    
    return filtered_files

def return_file_list_by_category(region_name, week_,obs_source):
    black = ['EX0','EX13']
    red = ['EX14','EX15','EX16','EX17','EX22','EX23','EX24','EX25']
    blue = ['EX1','EX2','EX3','EX4','EX5','EX6','EX7','EX8','EX9','EX10','EX11','EX12',
           'EX18','EX19','EX20','EX21','EX27','EX28','EX29']
    
    unet_files = sorted(glob(f'predictions/{region_name}/Wk{week_}_testing/*')) #With a specific subset of data
    #First find the correct experiments
    bias_correction_black_G = filter_files_by_ex_GEFS(unet_files, black, week_,obs_source)
    hybrid_blue_G = filter_files_by_ex_GEFS(unet_files, blue, week_,obs_source)
    obs_red_G = filter_files_by_ex_GEFS(unet_files, red, week_,obs_source)

    bias_correction_black_E = filter_files_by_ex_ECMWF(unet_files, black, week_,obs_source)
    hybrid_blue_E = filter_files_by_ex_ECMWF(unet_files, blue, week_,obs_source)
    obs_red_E = filter_files_by_ex_ECMWF(unet_files, red, week_,obs_source)
    
    return bias_correction_black_G, hybrid_blue_G, obs_red_G, bias_correction_black_E, hybrid_blue_E, obs_red_E


def create_empty_array(ecmwf_acc, day_num):
    u_acc = ecmwf_acc.sel(lead=day_num).copy(deep=True)
    u_crps = ecmwf_acc.sel(lead=day_num).copy(deep=True)
    u_acc[putils.xarray_varname(u_acc)][:,:] = 0
    u_crps[putils.xarray_varname(u_crps)][:,:] = 0
    return u_acc, u_crps

In [ ]:
def calculate_ACC_raw_forecast_all_seasons(obs_original, gefs, ecmwf):

    df_acc_full = pd.DataFrame()
    df_crps_full = pd.DataFrame()
    df_CRPS_full = pd.DataFrame()
    
    gefs_acc = verifications.create_climpred_ACC(verifications.rename_subx_for_climpred(gefs), verifications.rename_obs_for_climpred(obs_original))
    df_acc_full = add_lineplot_to_dataframe(df_acc_full,gefs_acc,'GEFSv12', 'ACC',10, 'mean') #Just keep the name the same for later

    ecmwf_acc = verifications.create_climpred_ACC(verifications.rename_subx_for_climpred(ecmwf), verifications.rename_obs_for_climpred(obs_original))
    df_acc_full = add_lineplot_to_dataframe(df_acc_full,gefs_acc,'ECMWF', 'ACC',10, 'mean') #Just keep the name the same for later

    gefs_CRPS = verifications.create_climpred_CRPS(verifications.rename_subx_for_climpred(gefs), verifications.rename_obs_for_climpred(obs_original))
    df_CRPS_full = add_lineplot_to_dataframe(df_crps_full,gefs_CRPS,'GEFSv12', 'CRPS',10, 'mean') #Just keep the name the same for later
    
    ecmwf_CRPS = verifications.create_climpred_CRPS(verifications.rename_subx_for_climpred(ecmwf), verifications.rename_obs_for_climpred(obs_original))
    df_CRPS_full = add_lineplot_to_dataframe(df_crps_full,gefs_CRPS,'ECMWF', 'CRPS',10, 'mean') #Just keep the name the same for later


    gefs_crps = verifications.create_climpred_CRPS(verifications.rename_subx_for_climpred(gefs), verifications.rename_obs_for_climpred(obs_original))
    df_crps_full = add_lineplot_to_dataframe(df_crps_full,gefs_CRPS,'GEFSv12', 'CRPS',10, 'mean') #Just keep the name the same for later
    
    ecmwf_crps = verifications.create_climpred_CRPS(verifications.rename_subx_for_climpred(ecmwf), verifications.rename_obs_for_climpred(obs_original))
    df_crps_full = add_lineplot_to_dataframe(df_crps_full,gefs_CRPS,'ECMWF', 'CRPS',10, 'mean') #Just keep the name the same for later

    
    return df_acc_full, df_crps_full, df_CRPS_full

In [ ]:


def calculate_ACC_raw_forecast_split_seasons(obs_original, gefs, ecmwf):
    df_acc_djf = pd.DataFrame()
    df_acc_mam = pd.DataFrame()
    df_acc_jja = pd.DataFrame()
    df_acc_son = pd.DataFrame()
    
    df_crps_djf = pd.DataFrame()
    df_crps_mam = pd.DataFrame()
    df_crps_jja = pd.DataFrame()
    df_crps_son = pd.DataFrame()

    df_CRPS_djf = pd.DataFrame()
    df_CRPS_mam = pd.DataFrame()
    df_CRPS_jja = pd.DataFrame()
    df_CRPS_son = pd.DataFrame()

    seasons = {
        "DJF": [12, 1, 2],  # Winter
        "MAM": [3, 4, 5],   # Spring
        "JJA": [6, 7, 8],   # Summer
        "SON": [9, 10, 11]  # Fall
    }

    for season, months in seasons.items():
        # break
        # Filter datasets for the current season
        obs_season = obs_original.sel(time=obs_original['time'].dt.month.isin(months))
        gefs_season = gefs.sel(S=gefs['S'].dt.month.isin(months))
        ecmwf_season = ecmwf.sel(S=ecmwf['S'].dt.month.isin(months))

        # Compute ACC and CRPS for the season
        gefs_acc = verifications.create_climpred_ACC(
            verifications.rename_subx_for_climpred(gefs_season), 
            verifications.rename_obs_for_climpred(obs_original)
        )
        ecmwf_acc = verifications.create_climpred_ACC(
            verifications.rename_subx_for_climpred(ecmwf_season), 
            verifications.rename_obs_for_climpred(obs_original)
        )

        gefs_CRPS = verifications.create_climpred_CRPS(
            verifications.rename_subx_for_climpred(gefs_season), 
            verifications.rename_obs_for_climpred(obs_original)
        )
        ecmwf_CRPS = verifications.create_climpred_CRPS(
            verifications.rename_subx_for_climpred(ecmwf_season), 
            verifications.rename_obs_for_climpred(obs_original)
        )

        gefs_crps = verifications.create_climpred_CRPS(
            verifications.rename_subx_for_climpred(gefs_season), 
            verifications.rename_obs_for_climpred(obs_original)
        )
        ecmwf_crps = verifications.create_climpred_CRPS(
            verifications.rename_subx_for_climpred(ecmwf_season), 
            verifications.rename_obs_for_climpred(obs_original)
        )

        if season == 'DJF':
            # Store results in DataFrame
            df_acc_djf = add_lineplot_to_dataframe(df_acc_djf, gefs_acc, 'GEFSv12', f'ACC', 10, 'mean')
            df_acc_djf = add_lineplot_to_dataframe(df_acc_djf, ecmwf_acc, 'ECMWF', f'ACC', 10, 'mean')
            df_crps_djf = add_lineplot_to_dataframe(df_crps_djf, gefs_crps, 'GEFSv12', f'CRPS', 10, 'mean')
            df_crps_djf = add_lineplot_to_dataframe(df_crps_djf, ecmwf_crps, 'ECMWF', f'CRPS', 10, 'mean')
            df_CRPS_djf = add_lineplot_to_dataframe(df_CRPS_djf, gefs_CRPS, 'GEFSv12', f'CRPS', 10, 'mean')
            df_CRPS_djf = add_lineplot_to_dataframe(df_CRPS_djf, ecmwf_CRPS, 'ECMWF', f'CRPS', 10, 'mean')
        elif season == 'MAM':
            df_acc_mam = add_lineplot_to_dataframe(df_acc_mam, gefs_acc, 'GEFSv12', f'ACC', 10, 'mean')
            df_acc_mam = add_lineplot_to_dataframe(df_acc_mam, ecmwf_acc, 'ECMWF', f'ACC', 10, 'mean')
            df_crps_mam = add_lineplot_to_dataframe(df_crps_mam, gefs_crps, 'GEFSv12', f'CRPS', 10, 'mean')
            df_crps_mam = add_lineplot_to_dataframe(df_crps_mam, ecmwf_crps, 'ECMWF', f'CRPS', 10, 'mean')
            df_CRPS_mam = add_lineplot_to_dataframe(df_CRPS_mam, gefs_CRPS, 'GEFSv12', f'CRPS', 10, 'mean')
            df_CRPS_mam = add_lineplot_to_dataframe(df_CRPS_mam, ecmwf_crps, 'ECMWF', f'CRPS', 10, 'mean')
        elif season == 'JJA':            
            df_acc_jja = add_lineplot_to_dataframe(df_acc_jja, gefs_acc, 'GEFSv12', f'ACC', 10, 'mean')
            df_acc_jja = add_lineplot_to_dataframe(df_acc_jja, ecmwf_acc, 'ECMWF', f'ACC', 10, 'mean')
            df_crps_jja = add_lineplot_to_dataframe(df_crps_jja, gefs_crps, 'GEFSv12', f'CRPS', 10, 'mean')
            df_crps_jja = add_lineplot_to_dataframe(df_crps_jja, ecmwf_crps, 'ECMWF', f'CRPS', 10, 'mean')
            df_CRPS_jja = add_lineplot_to_dataframe(df_CRPS_jja, gefs_CRPS, 'GEFSv12', f'CRPS', 10, 'mean')
            df_CRPS_jja = add_lineplot_to_dataframe(df_CRPS_jja, ecmwf_CRPS, 'ECMWF', f'CRPS', 10, 'mean')
        elif season == 'SON':
            df_acc_son = add_lineplot_to_dataframe(df_acc_son, gefs_acc, 'GEFSv12', f'ACC', 10, 'mean')
            df_acc_son = add_lineplot_to_dataframe(df_acc_son, ecmwf_acc, 'ECMWF', f'ACC', 10, 'mean')
            df_crps_son = add_lineplot_to_dataframe(df_crps_son, gefs_crps, 'GEFSv12', f'CRPS', 10, 'mean')
            df_crps_son = add_lineplot_to_dataframe(df_crps_son, ecmwf_crps, 'ECMWF', f'CRPS', 10, 'mean')
            df_CRPS_son = add_lineplot_to_dataframe(df_CRPS_son, gefs_CRPS, 'GEFSv12', f'CRPS', 10, 'mean')
            df_CRPS_son = add_lineplot_to_dataframe(df_CRPS_son, ecmwf_CRPS, 'ECMWF', f'CRPS', 10, 'mean')
               
    return df_acc_djf, df_crps_djf, df_acc_mam, df_crps_mam, df_acc_jja, df_crps_jja, df_acc_son, df_crps_son,df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son


In [ ]:
def return_blank_UNET_files(template,seasons):
    # Initialize all-season results
    e_acc, e_crps, e_CRPS = template.copy(deep=True), template.copy(deep=True), template.copy(deep=True)
    e_crps = e_crps.rename({'acc':'crps'})
    e_crps = e_CRPS.rename({'acc':'CRPS'})
    e_acc[putils.xarray_varname(e_acc)][:, :, :] = 0
    e_crps[putils.xarray_varname(e_crps)][:, :, :] = 0
    e_CRPS[putils.xarray_varname(e_CRPS)][:, :, :] = 0
    
    # Initialize dictionaries to store seasonal results
    seasonal_acc = {season: e_acc.copy(deep=True) for season in seasons}
    seasonal_crps = {season: e_crps.copy(deep=True) for season in seasons}
    seasonal_CRPS = {season: e_CRPS.copy(deep=True) for season in seasons}
    
    # Set initial values to zero for each season
    for season in seasons:
        seasonal_acc[season][putils.xarray_varname(e_acc)][:, :, :] = 0
        seasonal_crps[season][putils.xarray_varname(e_crps)][:, :, :] = 0
        seasonal_CRPS[season][putils.xarray_varname(e_CRPS)][:, :, :] = 0

    return e_acc, e_crps, e_CRPS, seasonal_acc, seasonal_crps, seasonal_CRPS


In [ ]:
def return_UNET_combined(region_name, test_start, test_end, obs_original, df_acc_full, df_crps_full, template, df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son, df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son,leads,
                         ecm_djf, gef_djf, ecm_mam, gef_mam, ecm_jja, gef_jja, ecm_son, gef_son,obs_source,df_CRPS_djf,df_CRPS_mam,df_CRPS_jja,df_CRPS_son,df_CRPS_full,
                        experiment_number = 'EX29'):
    
    # Define seasons
    seasons = f.return_seasons()
    e_acc, e_crps, e_CRPS, seasonal_acc, seasonal_crps, seasonal_CRPS = return_blank_UNET_files(template,seasons)
    
    obs_forecast_boolean = np.isnan(obs)
    
    #Add to a new gefs template for masking
    obs_forecast_boolean2 = gefs.copy(deep=True)
    obs_forecast_boolean2.RZSM[:,:,:,:,:] = obs_forecast_boolean[putils.xarray_varname(obs)]


    obs_original = xr.where(np.isnan(mask_anom),np.nan,obs_original)
    lead_improvment_plots = {}
    unet_crps_2d_ecmwf = {}
    unet_CRPS_2d_ecmwf = {}

    unet_crps_2d_gefsv12 = {}
    unet_CRPS_2d_gefsv12 = {}
    
    for idLead, week_ in enumerate([1, 2, 3, 4]):
        # break
        
        bias_correction_black_G, hybrid_blue_G, obs_red_G, bias_correction_black_E, hybrid_blue_E, obs_red_E  = return_file_list_by_category(region_name, week_,obs_source)
        day_num = (week_ * 7) - 1  # Lead time in days
    
        for model, name in zip([bias_correction_black_G, hybrid_blue_G, obs_red_G, bias_correction_black_E, hybrid_blue_E, obs_red_E], ['DM-BC_DL_GEFSv12', 'DL-DM_GEFSv12', 'DL_GEFSv12','DM-BC_DL_ECMWF', 'DL-DM_ECMWF', 'DL_ECMWF']):
            
            if len(model) !=0:
                # break
                print(f'Processing {name} for WEEK {week_}.')
                u_acc, u_crps, u_CRPS = e_acc.copy(deep=True).sel(lead=day_num), e_crps.copy(deep=True).sel(lead=day_num), e_CRPS.copy(deep=True).sel(lead=day_num)
        
                new_source = 'ECMWF' if 'ECMWF' in model else 'GEFSv12'
                final_exp = [i for i in model if experiment_number in i][0]
                test_name = final_exp.split('testing_')[-1].split('.npy')[0]
                
                # Load forecast file
                forecast = verifications.load_UNET_files(
                    gefs=gefs, file=model[0], region_name=region_name, day_num=day_num, 
                    new_source=new_source, test_year=test_year
                )
                
                # Apply land mask
                forecast = xr.where(obs_forecast_boolean2 == 0, forecast, np.nan)
                
                # Compute ACC and CRPS
                unet_acc = verifications.create_climpred_ACC(
                    verifications.rename_subx_for_climpred(forecast), 
                    verifications.rename_obs_for_climpred(obs_original)
                ).sel(lead=day_num)
    
                unet_crps = verifications.create_climpred_CRPS(
                    verifications.rename_subx_for_climpred(forecast), 
                    verifications.rename_obs_for_climpred(obs_original)
                ).sel(lead=day_num).mean(dim='init')

                unet_CRPS = verifications.create_climpred_CRPS(
                    verifications.rename_subx_for_climpred(forecast), 
                    verifications.rename_obs_for_climpred(obs_original)
                ).sel(lead=day_num).mean(dim='init')

        
                # Reapply mask
                mask = ~np.isnan(template[putils.xarray_varname(template)].sel(lead=day_num))
                mask_season = mask.rename({'lat':'Y','lon':'X'})
                mask_season = mask_season.drop_vars(['lead','skill'])
                u_acc = xr.where(mask, unet_acc, np.nan)
                u_crps = xr.where(mask, unet_crps, np.nan)
                u_CRPS = xr.where(mask, unet_CRPS, np.nan)
        
                # Add results to all-season DataFrame
                df_acc_full = add_lineplot_to_dataframe_average(df_acc_full, u_acc, name, 'ACC', week_, 'mean')
                df_crps_full = add_lineplot_to_dataframe_average(df_crps_full, u_crps, name, 'CRPS', week_, 'mean')
                df_CRPS_full = add_lineplot_to_dataframe_average(df_CRPS_full, u_CRPS, name, 'CRPS', week_, 'median')
                
                # Load forecast file
                forecast = verifications.load_UNET_files(
                    gefs=gefs, file=model[0], region_name=region_name, day_num=day_num, 
                    new_source=new_source, test_year=test_year
                )
                
                # Assign months
                forecast['month'] = forecast['S'].dt.month

                
                # Process seasonal data
                for season, months in seasons.items():
                    # break
                    
                    lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = {} 
                    
                    seasonal_forecast = forecast.where(forecast['month'].isin(months), drop=True)
                    # seasonal_forecast = xr.where(mask_season,seasonal_forecast,np.nan)
                    seasonal_obs = obs_original.sel(time=obs_original['time'].dt.month.isin(months)) #small subset
        
                    # Compute seasonal ACC and CRPS
                    season_acc = verifications.create_climpred_ACC(
                        verifications.rename_subx_for_climpred(seasonal_forecast),
                        verifications.rename_obs_for_climpred(seasonal_obs)
                    ).sel(lead=day_num)

                    '''Add improvments'''
                    if season == 'DJF':
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_djf.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_djf.isel(lead=idLead)
                            
                    elif season == "MAM":
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_mam.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_mam.isel(lead=idLead)
                    elif season == "JJA":
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_jja.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_jja.isel(lead=idLead)
                    elif season == "SON":
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_son.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_son.isel(lead=idLead)            
                    
                    season_crps = verifications.create_climpred_CRPS(
                        verifications.rename_subx_for_climpred(seasonal_forecast),
                        verifications.rename_obs_for_climpred(obs_original)
                    ).sel(lead=day_num).mean(dim='init')
                    
                    


                    season_CRPS = verifications.create_climpred_CRPS(
                        verifications.rename_subx_for_climpred(seasonal_forecast),
                        verifications.rename_obs_for_climpred(obs_original)
                    ).sel(lead=day_num).mean(dim='init')
                    
                    

                    #For some reason there are very extreme values
                    season_crps = xr.where(mask, season_crps, np.nan)

                    season_CRPS = xr.where(mask, season_CRPS, np.nan)
                    

                    if 'GEFSv12' in name:
                        unet_crps_2d_gefsv12[f'{season}_lead{week_}'] = season_crps
                        unet_CRPS_2d_gefsv12[f'{season}_lead{week_}'] = season_CRPS
                    else:
                        unet_crps_2d_ecmwf[f'{season}_lead{week_}'] = season_crps
                        unet_CRPS_2d_ecmwf[f'{season}_lead{week_}'] = season_CRPS
                    

                    
                    season_CRPS = xr.where(season_CRPS < -10, np.nan, season_CRPS)
                    # print(season_crps.mean())
                    # print(season_crps.median())
                    # print(season_CRPS.mean())
                    # print(season_CRPS.median())
                    # qp.d2_plot(season_crps.crps)
    
                    # Store seasonal results
                    seasonal_acc[season][putils.xarray_varname(seasonal_acc[season])][idLead,:,:] = season_acc[putils.xarray_varname(season_acc)]
                    seasonal_crps[season][putils.xarray_varname(seasonal_crps[season])][idLead,:,:]  = season_crps[putils.xarray_varname(season_crps)]
                    seasonal_CRPS[season][putils.xarray_varname(seasonal_CRPS[season])][idLead,:,:]  = season_CRPS[putils.xarray_varname(season_CRPS)]
        

    
                for season, months in seasons.items(): 
                    #winter
                    if season == 'DJF':
                        df_acc_djf = add_lineplot_to_dataframe_average_by_week(df_acc_djf, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_djf = add_lineplot_to_dataframe_average_by_week(df_CRPS_djf, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_djf = add_lineplot_to_dataframe_average_by_week(df_crps_djf, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                    elif season == "MAM":
                        df_acc_mam = add_lineplot_to_dataframe_average_by_week(df_acc_mam, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_mam = add_lineplot_to_dataframe_average_by_week(df_CRPS_mam, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_mam = add_lineplot_to_dataframe_average_by_week(df_crps_mam, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                    elif season == "JJA":
                        df_acc_jja = add_lineplot_to_dataframe_average_by_week(df_acc_jja, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_jja = add_lineplot_to_dataframe_average_by_week(df_CRPS_jja, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_jja = add_lineplot_to_dataframe_average_by_week(df_crps_jja, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                    elif season == "SON":
                        df_acc_son = add_lineplot_to_dataframe_average_by_week(df_acc_son, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_son = add_lineplot_to_dataframe_average_by_week(df_CRPS_son, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_son = add_lineplot_to_dataframe_average_by_week(df_crps_son, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
        
    print("Processing completed successfully!")
    return df_acc_full, df_crps_full, df_acc_djf, df_crps_djf, df_acc_mam, df_crps_mam, df_acc_jja, df_crps_jja, df_acc_son, df_crps_son,lead_improvment_plots,df_CRPS_full, df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son, unet_crps_2d_gefsv12, unet_CRPS_2d_gefsv12,unet_crps_2d_ecmwf, unet_CRPS_2d_ecmwf


In [ ]:
def return_UNET_combined_CONUS(region_name, test_start, test_end, obs_original, df_acc_full, df_crps_full, template, df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son, df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son,leads,
                         ecm_djf, gef_djf, ecm_mam, gef_mam, ecm_jja, gef_jja, ecm_son, gef_son,obs_source,df_CRPS_djf,df_CRPS_mam,df_CRPS_jja,df_CRPS_son,df_CRPS_full,
                        experiment_number = 'EX29'):
    
    # Define seasons
    seasons = f.return_seasons()
    e_acc, e_crps, e_CRPS, seasonal_acc, seasonal_crps, seasonal_CRPS = return_blank_UNET_files(template,seasons)
    
    obs_forecast_boolean = np.isnan(obs)
    
    #Add to a new gefs template for masking
    obs_forecast_boolean2 = gefs.copy(deep=True)
    obs_forecast_boolean2.RZSM[:,:,:,:,:] = obs_forecast_boolean[putils.xarray_varname(obs)]

    if region_name == 'CONUS':
        obs_original = xr.where(mask_anom==0,np.nan,obs_original)
    else:
        obs_original = xr.where(np.isnan(mask_anom),np.nan,obs_original)
        
    lead_improvment_plots = {}
    unet_crps_2d_ecmwf = {}
    unet_CRPS_2d_ecmwf = {}

    unet_crps_2d_gefsv12 = {}
    unet_CRPS_2d_gefsv12 = {}
    
    for idLead, week_ in enumerate([1, 2, 3, 4]):
        # break
        
        bias_correction_black_G, hybrid_blue_G, obs_red_G, bias_correction_black_E, hybrid_blue_E, obs_red_E  = return_file_list_by_category(region_name, week_,obs_source)
        day_num = (week_ * 7) - 1  # Lead time in days

        hybrid_blue_G = [i for i in hybrid_blue_G if "2019" not in i] 
        hybrid_blue_G = [i for i in hybrid_blue_G if "EX29_regular" in i]  #just keep a single model
        for model, name in zip([hybrid_blue_G,hybrid_blue_E], ['DL-DM_GEFSv12','DL-DM_ECMWF']):

            if len(model) !=0:

                print(f'Processing {name} for WEEK {week_}.')
                u_acc, u_crps, u_CRPS = e_acc.copy(deep=True).sel(lead=day_num), e_crps.copy(deep=True).sel(lead=day_num), e_CRPS.copy(deep=True).sel(lead=day_num)
        
                new_source = 'ECMWF' if 'ECMWF' in model else 'GEFSv12'
                final_exp = [i for i in model if experiment_number in i][0]
                test_name = final_exp.split('testing_')[-1].split('.npy')[0]
                
                # Load forecast file
                forecast = verifications.load_UNET_files(
                    gefs=gefs, file=model[0], region_name=region_name, day_num=day_num, 
                    new_source=new_source, test_year=test_year
                )
                
                # Apply land mask
                forecast = xr.where(obs_forecast_boolean2 == 0, forecast, np.nan)
                
                # Compute ACC and CRPS
                unet_acc = verifications.create_climpred_ACC(
                    verifications.rename_subx_for_climpred(forecast), 
                    verifications.rename_obs_for_climpred(obs_original)
                ).sel(lead=day_num)
    
                unet_crps = verifications.create_climpred_CRPS(
                    verifications.rename_subx_for_climpred(forecast), 
                    verifications.rename_obs_for_climpred(obs_original)
                ).sel(lead=day_num).mean(dim='init')

                unet_CRPS = verifications.create_climpred_CRPS(
                    verifications.rename_subx_for_climpred(forecast), 
                    verifications.rename_obs_for_climpred(obs_original)
                ).sel(lead=day_num).mean(dim='init')

        
                # Reapply mask
                mask = ~np.isnan(template[putils.xarray_varname(template)].sel(lead=day_num))
                mask_season = mask.rename({'lat':'Y','lon':'X'})
                mask_season = mask_season.drop_vars(['lead','skill'])
                u_acc = xr.where(mask, unet_acc, np.nan)
                u_crps = xr.where(mask, unet_crps, np.nan)
                u_CRPS = xr.where(mask, unet_CRPS, np.nan)
        
                # Add results to all-season DataFrame
                df_acc_full = add_lineplot_to_dataframe_average(df_acc_full, u_acc, name, 'ACC', week_, 'mean')
                df_crps_full = add_lineplot_to_dataframe_average(df_crps_full, u_crps, name, 'CRPS', week_, 'mean')
                df_CRPS_full = add_lineplot_to_dataframe_average(df_CRPS_full, u_CRPS, name, 'CRPS', week_, 'median')
                
                # Load forecast file
                forecast = verifications.load_UNET_files(
                    gefs=gefs, file=model[0], region_name=region_name, day_num=day_num, 
                    new_source=new_source, test_year=test_year
                )
                
                # Assign months
                forecast['month'] = forecast['S'].dt.month

                
                # Process seasonal data
                for season, months in seasons.items():
                    # break
                    
                    lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = {} 
                    
                    seasonal_forecast = forecast.where(forecast['month'].isin(months), drop=True)
                    # seasonal_forecast = xr.where(mask_season,seasonal_forecast,np.nan)
                    seasonal_obs = obs_original.sel(time=obs_original['time'].dt.month.isin(months)) #small subset
        
                    # Compute seasonal ACC and CRPS
                    season_acc = verifications.create_climpred_ACC(
                        verifications.rename_subx_for_climpred(seasonal_forecast),
                        verifications.rename_obs_for_climpred(seasonal_obs)
                    ).sel(lead=day_num)

                    '''Add improvments'''
                    if season == 'DJF':
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_djf.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_djf.isel(lead=idLead)
                            
                    elif season == "MAM":
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_mam.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_mam.isel(lead=idLead)
                    elif season == "JJA":
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_jja.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_jja.isel(lead=idLead)
                    elif season == "SON":
                        if 'GEFSv12' in name:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - gef_son.isel(lead=idLead)
                        else:
                            lead_improvment_plots[f'{name}_lead{week_}_season{season}'] = season_acc - ecm_son.isel(lead=idLead)            
                    
                    season_crps = verifications.create_climpred_CRPS(
                        verifications.rename_subx_for_climpred(seasonal_forecast),
                        verifications.rename_obs_for_climpred(obs_original)
                    ).sel(lead=day_num).mean(dim='init')
                    
                    


                    season_CRPS = verifications.create_climpred_CRPS(
                        verifications.rename_subx_for_climpred(seasonal_forecast),
                        verifications.rename_obs_for_climpred(obs_original)
                    ).sel(lead=day_num).mean(dim='init')
                    
                    

                    #For some reason there are very extreme values
                    season_crps = xr.where(mask, season_crps, np.nan)

                    season_CRPS = xr.where(mask, season_CRPS, np.nan)
                    

                    if 'GEFSv12' in name:
                        unet_crps_2d_gefsv12[f'{season}_lead{week_}'] = season_crps
                        unet_CRPS_2d_gefsv12[f'{season}_lead{week_}'] = season_CRPS
                    else:
                        unet_crps_2d_ecmwf[f'{season}_lead{week_}'] = season_crps
                        unet_CRPS_2d_ecmwf[f'{season}_lead{week_}'] = season_CRPS
                    

                    
                    season_CRPS = xr.where(season_CRPS < -10, np.nan, season_CRPS)
                    # print(season_crps.mean())
                    # print(season_crps.median())
                    # print(season_CRPS.mean())
                    # print(season_CRPS.median())
                    # qp.d2_plot(season_crps.crps)
    
                    # Store seasonal results
                    seasonal_acc[season][putils.xarray_varname(seasonal_acc[season])][idLead,:,:] = season_acc[putils.xarray_varname(season_acc)]
                    seasonal_crps[season][putils.xarray_varname(seasonal_crps[season])][idLead,:,:]  = season_crps[putils.xarray_varname(season_crps)]
                    seasonal_CRPS[season][putils.xarray_varname(seasonal_CRPS[season])][idLead,:,:]  = season_CRPS[putils.xarray_varname(season_CRPS)]
        

    
                for season, months in seasons.items(): 
                    #winter
                    if season == 'DJF':
                        df_acc_djf = add_lineplot_to_dataframe_average_by_week(df_acc_djf, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_djf = add_lineplot_to_dataframe_average_by_week(df_CRPS_djf, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_djf = add_lineplot_to_dataframe_average_by_week(df_crps_djf, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                    elif season == "MAM":
                        df_acc_mam = add_lineplot_to_dataframe_average_by_week(df_acc_mam, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_mam = add_lineplot_to_dataframe_average_by_week(df_CRPS_mam, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_mam = add_lineplot_to_dataframe_average_by_week(df_crps_mam, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                    elif season == "JJA":
                        df_acc_jja = add_lineplot_to_dataframe_average_by_week(df_acc_jja, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_jja = add_lineplot_to_dataframe_average_by_week(df_CRPS_jja, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_jja = add_lineplot_to_dataframe_average_by_week(df_crps_jja, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                    elif season == "SON":
                        df_acc_son = add_lineplot_to_dataframe_average_by_week(df_acc_son, seasonal_acc[season], name, 'ACC', 10, 'mean',day_num, idLead)
                        df_CRPS_son = add_lineplot_to_dataframe_average_by_week(df_CRPS_son, seasonal_CRPS[season], name, 'CRPS', 10, 'mean',day_num, idLead)
                        df_crps_son = add_lineplot_to_dataframe_average_by_week(df_crps_son, seasonal_crps[season], name, 'CRPS', 10, 'mean',day_num, idLead)
        
    print("Processing completed successfully!")
    return df_acc_full, df_crps_full, df_acc_djf, df_crps_djf, df_acc_mam, df_crps_mam, df_acc_jja, df_crps_jja, df_acc_son, df_crps_son,lead_improvment_plots,df_CRPS_full, df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son, unet_crps_2d_gefsv12, unet_CRPS_2d_gefsv12,unet_crps_2d_ecmwf, unet_CRPS_2d_ecmwf


In [ ]:


leads=[6,13,20,27]

print(f'Calculating ACC and CRPS on GEFS and ECMWF')
obs, gefs, ecmwf = obs_anomaly_SubX_format.sel(L=leads), baseline_gefs.sel(L=leads), baseline_ecmwf.sel(L=leads)

try:
    obs_original = obs_original.rename({'X':'longitude','Y':'latitude'})
except ValueError:
    obs_original = obs_original
template = verifications.create_climpred_ACC(verifications.rename_subx_for_climpred(ecmwf), verifications.rename_obs_for_climpred(obs_original))

'''All seasons raw forecast metrics'''
df_acc_full, df_crps_full, df_CRPS_full = calculate_ACC_raw_forecast_all_seasons(obs_original, gefs, ecmwf)

'''Individual seasons raw forecast metrics. ACC and CRPS. These are already averaged over both raw datasets'''
df_acc_djf, df_crps_djf, df_acc_mam, df_crps_mam, df_acc_jja, df_crps_jja, df_acc_son, df_crps_son, df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son = calculate_ACC_raw_forecast_split_seasons(obs_original, gefs, ecmwf) 

# #Save bias corrected results to a dictionary for later plotting (to see improvement)
# ecmwf_bc_crps_2d = {}
# gefsv12_bc_crps_2d = {}


# seasons = f.return_seasons()
# for season, months in seasons.items():
#     if season == 'DJF':
#         ecmwf_bc_crps_2d[season] = ecm_BC_djf.mean(dim='init')
#         gefsv12_bc_crps_2d[season] = gef_BC_djf.mean(dim='init')
#     elif season == 'MAM':
#         ecmwf_bc_crps_2d[season] = ecm_BC_mam.mean(dim='init')
#         gefsv12_bc_crps_2d[season] = gef_BC_mam.mean(dim='init')
#     elif season == 'JJA':
#         ecmwf_bc_crps_2d[season] = ecm_BC_jja.mean(dim='init')
#         gefsv12_bc_crps_2d[season] = gef_BC_jja.mean(dim='init')        
#     elif season == 'SON':
#         ecmwf_bc_crps_2d[season] = ecm_BC_son.mean(dim='init')
#         gefsv12_bc_crps_2d[season] = gef_BC_son.mean(dim='init')  

In [ ]:

'''Load bias corrected ACC all seasons (metrics already computed), Need these as a template'''
gef_BC, ecm_BC = loadbias.load_additive_bias_corrected_data_ACC(leads,region_name,obs_source)

#Reload the ACC
ecm_BC_djf, gef_BC_djf, ecm_BC_mam, gef_BC_mam, ecm_BC_jja, gef_BC_jja, ecm_BC_son, gef_BC_son = loadbias.load_additive_bias_corrected_data_by_season(leads,region_name,'acc',obs_source) 
'''UNET all seasons and each individual season. ACC and CRPS'''


#We are taking the average of all models
'''also compute the ACC improvement over each of the seasons'''
if region_name !='CONUS':
    df_acc_full, df_crps_full, df_acc_djf, df_crps_djf, df_acc_mam, df_crps_mam, df_acc_jja, df_crps_jja, df_acc_son, df_crps_son,lead_improvment_plots,df_CRPS_full, df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son,unet_crps_2d_gefsv12, unet_CRPS_2d_gefsv12,unet_crps_2d_ecmwf, unet_CRPS_2d_ecmwf = return_UNET_combined(region_name, test_start, test_end, obs_original, df_acc_full, df_crps_full, template, df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son, df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son,leads,
                             ecm_BC_djf, gef_BC_djf, ecm_BC_mam, gef_BC_mam, ecm_BC_jja, gef_BC_jja, ecm_BC_son, gef_BC_son,obs_source,df_CRPS_djf,df_CRPS_mam,df_CRPS_jja,df_CRPS_son, df_CRPS_full)
else:
    df_acc_full, df_crps_full, df_acc_djf, df_crps_djf, df_acc_mam, df_crps_mam, df_acc_jja, df_crps_jja, df_acc_son, df_crps_son,lead_improvment_plots,df_CRPS_full, df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son,unet_crps_2d_gefsv12, unet_CRPS_2d_gefsv12,unet_crps_2d_ecmwf, unet_CRPS_2d_ecmwf = return_UNET_combined_CONUS(region_name, test_start, test_end, obs_original, df_acc_full, df_crps_full, template, df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son, df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son,leads,
                                 ecm_BC_djf, gef_BC_djf, ecm_BC_mam, gef_BC_mam, ecm_BC_jja, gef_BC_jja, ecm_BC_son, gef_BC_son,obs_source,df_CRPS_djf,df_CRPS_mam,df_CRPS_jja,df_CRPS_son, df_CRPS_full)
    


In [ ]:
def find_global_max_min(datasets, metric):
    max_, min_ = [], []
    for idx,i in enumerate(datasets):
        min_.append(i[metric].min())
        max_.append(i[metric].max())
    return max(max_), min(min_)
        

## ACC and CRPS lineplots (2x2)

In [ ]:

def plot_ACC_CRPS_LinePlot(df_acc_full, df_CRPS_full, acc_datasets, CRPS_datasets,obs_source, metric_):
    save_dir = f'Outputs/ACC_CRPS_line_plots/{region_name}'
    os.system(f'mkdir -p {save_dir}')
    
    # Define datasets for ACC (first row) and CRPS (second row)
    season_names = ['DJF', 'MAM', 'JJA', 'SON']
    
    acc_max, acc_min = find_global_max_min(acc_datasets, 'ACC')
    CRPS_max, CRPS_min = find_global_max_min(CRPS_datasets, metric_)
    
    #manually change the max acc to 1 and add a small increment to the bottom
    acc_max = 0.9
    acc_min =-0.1

    if metric_ == 'crpss':
        CRPS_min = CRPS_min-0.2
        CRPS_max = CRPS_max+0.2
    else:
        CRPS_min = CRPS_min-0.002
        CRPS_max = CRPS_max+0.002
    
    
    #Plot
    fig, axs = plt.subplots(2,4,figsize=(20, 7),dpi=300)
    plt.style.use('seaborn-v0_8-colorblind')
    palette = plt.get_cmap('tab10')
    
    # fig.suptitle('ACC and CRPS by Season', fontsize=16)
    
    for (row,df),metric in zip(enumerate([acc_datasets,CRPS_datasets]),['ACC',metric_]):
        print(metric)
        for col,data in enumerate(df):
            df = data.copy()  # <-- safe!
            grouped = df.groupby('Forecast')
            color_tracker = {}
            marker_style = ['o', 'v', '^', '<', '>', 's', 'p', '*', '+']
            season = season_names[col]
            for i, (name, group) in enumerate(grouped):
                color = palette(i)
                marker = marker_style[i % len(marker_style)]
                if color not in color_tracker:
                    axs[row, col].plot(group['Week'], group[metric], label=custom_names[name], color=color, marker=marker, linestyle='-', markersize=6)
                    color_tracker[color] = custom_names[name]
                else:
                    axs[row, col].plot(group['Week'], group[metric], label=custom_names[name], color = color, marker=marker, linestyle='-', markersize=6)           
                
                axs[row, col].set_ylim(acc_min, acc_max)
                axs[row, col].set_ylim(acc_min if row == 0 else CRPS_min, acc_max if row == 0 else CRPS_max)
    
                # Add a horizontal line at y=0.5
                if row == 0:
                    axs[row,col].axhline(y=0.5, color='gray', linestyle='--', linewidth=2)
                    axs[row, col].set_title(season, fontsize=20)
            
                # Setting the title and labels with increased font sizes
                # plt.title(metric, fontsize=30)
                if row==1:
                    axs[row, col].set_xlabel('Week Lead', fontsize=14)
                if col ==0:
                    axs[row, col].set_ylabel(metric, fontsize=22, labelpad=10)

                if row == 0 and col == 0:
                    axs[row,col].legend(title='Forecast (# models)', fontsize=10, title_fontsize=13, loc='lower left') # Create the legend with a slightly larger font size
        
                axs[row, col].set_xticks([1,2,3,4])
                axs[row, col].tick_params(axis='both', which='major', labelsize=12)
                axs[row, col].grid(True, which='both', linestyle='--', linewidth=0.5) # Add a grid for better readability
    # Adjust layout and show plot
    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust for suptitle spacing
    plt.savefig(f'{save_dir}/Lineplot_ACC_{metric_}_SEASON_single_experiments_NO_BIAS_correction_{obs_source}.png', dpi=300)
    plt.show()
    return 0


#CRPS
plot_ACC_CRPS_LinePlot(df_acc_full, df_CRPS_full,\
                    acc_datasets = [df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son],\
                       CRPS_datasets = [df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son],\
                       obs_source=obs_source, metric_ = 'CRPS')

#CRPSS
# plot_ACC_CRPS_LinePlot(df_acc_full, df_crps_full,\
#                     acc_datasets = [df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son],\
#                        CRPS_datasets = [df_crpss_djf, df_crpss_mam, df_crpss_jja, df_crpss_son],\
#                        obs_source=obs_source, metric_ = 'CRPSS')


# Now plot with the improvement ACC data in them

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib.cm import get_cmap

def plot_combined_ACC_CRPS_map_grid(df_acc_list, df_crps_list, forecast_dict, forecast_name, variable_name, cmap,obs_source, metric_):
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    leads = [1, 2, 3, 4]
    marker_style = ['o', 'v', '^', '<', '>', 's', 'p', '*', '+']
    palette = plt.get_cmap('tab10')
    
    
    acc_max, acc_min = find_global_max_min(df_acc_list, 'ACC')
    CRPS_max, CRPS_min = find_global_max_min(df_crps_list, metric_)
    
    #manually change the max acc to 1 and add a small increment to the bottom
    acc_max = 0.9
    acc_min =-0.1

    if metric_ == 'CRPSS':
        CRPS_min = CRPS_min-0.2
        CRPS_max = CRPS_max+0.2
    else:
        CRPS_min = CRPS_min-0.002
        CRPS_max = CRPS_max+0.002

    fig, axs = plt.subplots(6, 4, figsize=(22, 20), dpi=300,)  # We'll override projection for maps later
    # ✅ Inject Cartopy projection only in bottom 4 rows
    for row in range(2, 6):
        for col in range(4):
            axs[row, col].remove()
            axs[row, col] = fig.add_subplot(6, 4, row * 4 + col + 1, projection=ccrs.PlateCarree())
    
    # ------------------ Line Plots (Top 2 Rows) ------------------
    for row_idx, (df_list, metric) in enumerate(zip([df_acc_list, df_crps_list], ['ACC', metric_])):
        for col_idx, data in enumerate(df_list):
            df = data.copy()  # <-- safe!
            grouped = df.groupby('Forecast')
            ax = axs[row_idx, col_idx]
    
            for i, (name, group) in enumerate(grouped):
                color = palette(i)
                marker = marker_style[i % len(marker_style)]
                ax.plot(group['Week'], group[metric], label=name, color=color,
                        marker=marker, linestyle='-', markersize=6)
    
            ax.set_xlim(0.5, 4.5)
            ax.set_xticks([1, 2, 3, 4])
            ax.set_xticklabels(['1', '2', '3', '4'])
            ax.xaxis.set_major_locator(ticker.FixedLocator([1, 2, 3, 4]))
    
            ax.grid(True, linestyle='--', linewidth=0.5)
            ax.tick_params(labelsize=10)
            ax.set_ylim(acc_min if row_idx == 0 else CRPS_min, acc_max if row_idx == 0 else CRPS_max)
    
            if row_idx == 0:
                ax.axhline(0.5, linestyle='--', color='gray', linewidth=1.5)
                ax.set_title(seasons[col_idx], fontsize=20)
                # ax.set_title(seasons[col_idx], fontsize=20, fontweight='bold')
            if row_idx == 1:
                ax.set_xlabel('Lead Week', fontsize=12)
            if col_idx == 0:
                ax.set_ylabel(metric, fontsize=14)
    
            if row_idx == 0 and col_idx == 0:
                ax.legend(title='Forecast', fontsize=9, title_fontsize=10)
    
    # ------------------ Compute global color scale for maps ------------------
    vmin_map = np.inf
    vmax_map = -np.inf
    for key in forecast_dict:
        if variable_name in forecast_dict[key].data_vars:
            data = forecast_dict[key][variable_name].values
            vmin_map = min(vmin_map, np.nanmin(data))
            vmax_map = max(vmax_map, np.nanmax(data))
    
    vmin_map = np.floor(vmin_map * 10) / 10
    vmax_map = np.ceil(vmax_map * 10) / 10
    cmap = get_cmap(cmap)
    norm = TwoSlopeNorm(vmin=vmin_map, vcenter=0, vmax=vmax_map)
    if metric_ == 'CRPS':
        levels = np.arange(vmin_map, vmax_map , 0.2)
    
    
    # ------------------ Map Plots (Bottom 4 Rows) ------------------
    for row_idx, lead in enumerate(leads, start=2):
        for col_idx, season in enumerate(seasons):
            key = f'DL-DM_{forecast_name}_lead{lead}_season{season}'
            ax = axs[row_idx, col_idx]
            ax.set_aspect('auto')
    
            if key not in forecast_dict or variable_name not in forecast_dict[key].data_vars:
                ax.set_title(f'{key}\n(No data)', fontsize=10)
                ax.coastlines()
                continue
    
            ds = forecast_dict[key]
            data = ds[variable_name]
            lon = ds.lon.values
            lat = ds.lat.values
            mesh_lon, mesh_lat = np.meshgrid(lon, lat)
    
            im = ax.contourf(mesh_lon, mesh_lat, data.values,
                             levels=levels,
                             transform=ccrs.PlateCarree(), cmap=cmap, norm=norm, extend='both')
    
    
            ax.coastlines()
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.set_title(f'Season: {season}, Week: {lead}', fontsize=14)
            # ax.set_title(f'Season: {season}, Week: {lead}', fontsize=12, fontweight='bold')

    
            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xformatter = LongitudeFormatter()
            gl.yformatter = LatitudeFormatter()
            gl.xlabel_style = {'size': 8}
            gl.ylabel_style = {'size': 8}
    
    
    plt.tight_layout(rect=[0, 0, 0.93, 1])  # Adjust for suptitle spacing
    # Update ScalarMappable (same as before)
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    
    # Colorbar axis
    cbar_ax = fig.add_axes([0.945, 0.12, 0.015, 0.50])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='both',
                        boundaries=levels, ticks=levels)
    # 0.92, 0.08, 0.015, 0.40
    # Labels and appearance
    cbar.set_label(f'ACC Improvement diff(EX29-{forecast_name})', fontsize=14, rotation=90, labelpad=5)
    cbar.ax.tick_params(labelsize=14, pad=2)

    row_labels = ['A', 'B', 'C', 'D', 'E', 'F']
    # row_labels = ['(A)', '(B)', '(C)', '(D)', '(E)', '(F)']
    for row_idx, label in enumerate(row_labels):
        axs[row_idx, 0].text(
            -0.09, 1.05,  # ← coordinates (x, y) relative to that subplot
            label, 
            transform=axs[row_idx, 0].transAxes,  # ← place text in subplot's coordinate system
            fontsize=20, fontweight='bold', va='top', ha='right'
        )

    
    plt.savefig(f'Outputs/ACC_CRPS_line_plots/{region_name}/Combined_6x4_ACC_{metric_}_{forecast_name}_{obs_source}_NO_BIAS_corrected_single_experiment.png', dpi=300)
    plt.show()

In [ ]:
for forecast_name in ['ECMWF','GEFSv12']:
    # Filtered dictionary: only keep keys that contain 'GEFsv12'
    gefs_dict = {k: v for k, v in lead_improvment_plots.items() if 'GEFSv12' in k}
    ecmwf_dict = {k: v for k, v in lead_improvment_plots.items() if 'ECMWF' in k}
    if forecast_name == 'GEFSv12':
        forecast_dict = gefs_dict
    else:
        forecast_dict = ecmwf_dict

    plot_combined_ACC_CRPS_map_grid(
        df_acc_list=[df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son],
        df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son],
        metric_ = 'CRPS',
        forecast_dict=forecast_dict,
        forecast_name=forecast_name,
        variable_name='acc',
        cmap='bwr',
        obs_source=obs_source
    )
    
    # plot_combined_ACC_CRPS_map_grid(
    #     df_acc_list=[df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son],
    #     df_crps_list=[df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son],
    #     metric_ = 'CRPSS',
    #     forecast_dict=forecast_dict,
    #     forecast_name=forecast_name,
    #     variable_name='acc',
    #     cmap='bwr',
    #     obs_source=obs_source
    # )



# Plot ACC only

In [ ]:
def plot_combined_ACC_map_grid(df_acc_list, forecast_dict, forecast_name, variable_name, cmap, obs_source, region_name):
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    leads = [1, 2, 3, 4]
    marker_style = ['o', 'v', '^', '<', '>', 's', 'p', '*', '+']
    palette = plt.get_cmap('tab10')

    fig, axs = plt.subplots(5, 4, figsize=(24, 20), dpi=300, gridspec_kw={'hspace': 0.5})

    # Get ACC limits
    acc_max, acc_min = find_global_max_min(df_acc_list, 'ACC')
    acc_max = 1.0
    acc_min = -0.1

    # ✅ Inject Cartopy projection only in bottom 4 rows (rows 1–4)
    for row in range(1, 5):
        for col in range(4):
            axs[row, col].remove()
            axs[row, col] = fig.add_subplot(5, 4, row * 4 + col + 1, projection=ccrs.PlateCarree())

    # ------------------ ACC Line Plots (Top Row Only) ------------------
    for col_idx, data in enumerate(df_acc_list):
        ax = axs[0, col_idx]
        df = data.copy()  # <-- safe!
        grouped = df.groupby('Forecast')


        for i, (name, group) in enumerate(grouped):
            color = palette(i)
            marker = marker_style[i % len(marker_style)]
            ax.plot(group['Week'], group['ACC'], label=name, color=color,
                    marker=marker, linestyle='-', markersize=6)

        ax.set_xlim(0.5, 4.5)
        ax.set_xticks([1, 2, 3, 4])
        ax.set_xticklabels(['1', '2', '3', '4'])
        ax.xaxis.set_major_locator(ticker.FixedLocator([1, 2, 3, 4]))
        ax.grid(True, linestyle='--', linewidth=0.5)
        ax.tick_params(labelsize=16)
        ax.set_ylim(acc_min, acc_max)
        ax.axhline(0.5, linestyle='--', color='gray', linewidth=1.5)
        ax.set_title(seasons[col_idx], fontsize=21)
        ax.set_xlabel('Lead Week', fontsize=16)
        if col_idx == 0:
            ax.set_ylabel('ACC', fontsize=16)
            ax.legend(title='Forecast', fontsize=9, title_fontsize=10, loc='lower left')

    # ------------------ Compute global color scale for maps ------------------
    vmin_map = np.inf
    vmax_map = -np.inf
    for key in forecast_dict:
        if variable_name in forecast_dict[key].data_vars:
            data = forecast_dict[key][variable_name].values
            vmin_map = min(vmin_map, np.nanmin(data))
            vmax_map = max(vmax_map, np.nanmax(data))

    vmin_map = np.floor(vmin_map * 10) / 10
    vmax_map = np.ceil(vmax_map * 10) / 10
    cmap = colormaps[cmap]
    norm = TwoSlopeNorm(vmin=vmin_map, vcenter=0, vmax=vmax_map)
    levels = np.arange(vmin_map, vmax_map, 0.2)

    # ------------------ Map Plots (Rows 1–4) ------------------
    for row_idx, lead in enumerate(leads, start=1):
        for col_idx, season in enumerate(seasons):
            key = f'DL-DM_{forecast_name}_lead{lead}_season{season}'
            ax = axs[row_idx, col_idx]
            ax.set_aspect('auto')

            if key not in forecast_dict or variable_name not in forecast_dict[key].data_vars:
                ax.set_title(f'{key}\n(No data)', fontsize=17)
                ax.coastlines()
                continue

            ds = forecast_dict[key]
            data = ds[variable_name]
            lon = ds.lon.values
            lat = ds.lat.values
            mesh_lon, mesh_lat = np.meshgrid(lon, lat)

            im = ax.contourf(mesh_lon, mesh_lat, data.values,
                             levels=levels,
                             transform=ccrs.PlateCarree(), cmap=cmap, norm=norm, extend='both')

            ax.coastlines()
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.set_title(f'Season: {season}, Week: {lead}', fontsize=17)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xformatter = LongitudeFormatter()
            gl.yformatter = LatitudeFormatter()
            gl.xlabel_style = {'size': 13}
            gl.ylabel_style = {'size': 13}

    plt.tight_layout(rect=[0, 0, 0.93, 1])

    # Colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.92, 0.11, 0.020, 0.60])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='both',
                        boundaries=levels, ticks=levels)
    cbar.set_label(f'ACC Improvement diff(EX29-{forecast_name})', fontsize=18, rotation=90, labelpad=5)
    cbar.ax.tick_params(labelsize=14, pad=2)

    # Row labels
    row_labels = ['A', 'B', 'C', 'D', 'E']
    for row_idx, label in enumerate(row_labels):
        axs[row_idx, 0].text(
            -0.13, 1.12,
            label,
            transform=axs[row_idx, 0].transAxes,
            fontsize=22, fontweight='bold', va='top', ha='right'
        )

    plt.savefig(f'Outputs/ACC_CRPS_line_plots/{region_name}/Combined_5x4_ACC_only_{forecast_name}_{obs_source}_NO_BIAS_corrected_single_experiment.png', dpi=300)
    plt.show()


In [ ]:
for forecast_name in ['ECMWF','GEFSv12']:
    if forecast_name == 'GEFSv12':
        forecast_dict = gefs_dict
    else:
        forecast_dict = ecmwf_dict
    plot_combined_ACC_map_grid(
        df_acc_list=[df_acc_djf, df_acc_mam, df_acc_jja, df_acc_son],
        forecast_dict=forecast_dict,
        forecast_name=forecast_name,
        variable_name='acc',
        cmap='bwr',
        obs_source=obs_source,
        region_name=region_name
    )



# Plot where the CRPS is improved over bias corrected forecast (no simulations)

In [ ]:
def plot_combined_CRPS_greater_than_0(df_crps_list, df_crps_2d, forecast_name, obs_source, region_name, metric_, forecast_dict,better_than):
    '''gridded CRPS maps using df_crps_2d
    
    Only locations where the CRPS > 0 are kept. 
    
    If reforecast bias corrected <0 and UNET >0, return as value of 1. Else there is no improvement
    We are'''


    new_dir = f'Outputs/CRPS_improvement_plots/{region_name}'
    os.makedirs(new_dir,exist_ok=True)
    
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    leads = [1, 2, 3, 4]

    fig, axs = plt.subplots(
    4, 4, 
    figsize=(24, 17), 
    dpi=300, 
    subplot_kw={"projection": ccrs.PlateCarree()}
    )

    cmap = mcolors.ListedColormap(['#ffffff', '#1f77b4'])  # white for nan, blue for 1
    bounds = [0, 0.5, 1]  # boundaries between nan and 1
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    # ------------------ CRPS Map Plots (Rows 1–4) ------------------
    for row_idx, lead in enumerate(leads, start=0):
        for col_idx, season in enumerate(seasons):
            key = f'{season}_lead{row_idx+1}'
            ax = axs[row_idx, col_idx]
            ax.set_aspect('auto')

            f_key = putils.xarray_varname(forecast_dict[season])
            bc_final = xr.where(forecast_dict[season]>0,1,0).isel(lead=row_idx)[f_key]

            name = putils.xarray_varname(df_crps_2d[key])
            ds = df_crps_2d[key]
            data = ds[name]
            lon = ds.lon.values
            lat = ds.lat.values
            mesh_lon, mesh_lat = np.meshgrid(lon, lat)

            #Mask greater than 0
            unet_final = xr.where(data>0,1,np.nan)

            #This is where we see improvements in the forecast
            if better_than == True:
                unet_final = xr.where((unet_final == 1) & (bc_final == 0), unet_final, np.nan)

            im = ax.pcolormesh(mesh_lon, mesh_lat, unet_final,
                               transform=ccrs.PlateCarree(),
                               cmap=cmap, norm=norm)

            ax.coastlines()
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.set_title(f'Season: {season}, Week: {lead}', fontsize=14)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xformatter = LongitudeFormatter()
            gl.yformatter = LatitudeFormatter()
            gl.xlabel_style = {'size': 8}
            gl.ylabel_style = {'size': 8}

    if better_than == True:
        suptitle_text = f"CRPS Improvement over Bias-corrected {forecast_name}\nWhere UNET > 0 and {forecast_name} < 0"
    else:
        suptitle_text = f"CRPS greater than 0"
    
    plt.suptitle(suptitle_text, fontsize=30, y=0.97)
    plt.tight_layout(rect=[0, 0, 1, 0.93])  # Reserve space for the suptitle
    # Row labels (A–E)
    row_labels = ['A', 'B', 'C', 'D']
    for row_idx, label in enumerate(row_labels):
        axs[row_idx, 0].text(
            -0.09, 1.05,
            label,
            transform=axs[row_idx, 0].transAxes,
            fontsize=20, fontweight='bold', va='top', ha='right'
        )

    if better_than == True:
        save_file = f'{new_dir}/better_than_bias_corrected_{forecast_name}_{obs_source}_Map.png'
    else:
        save_file = f'{new_dir}/greater_than_0_{forecast_name}_{obs_source}_Map.png'
        
    plt.savefig(save_file, dpi=300)
    plt.show()


In [ ]:
# for better_than in [True,False]:
#     for forecast_name in ['ECMWF','GEFSv12']:
#         metric_ = 'CRPS'
#         if forecast_name == 'GEFSv12' and metric_=='CRPS':
#             forecast_dict = gefsv12_bc_CRPS_2d
#             unet_file = unet_CRPS_2d_gefsv12
#             df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
#         elif forecast_name == 'ECMWF' and metric_=='CRPS':
#             forecast_dict = ecmwf_bc_CRPS_2d
#             unet_file = unet_CRPS_2d_ecmwf
#             df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
    
    
#         plot_combined_CRPS_greater_than_0(
#             df_crps_list=df_crps_list,
#             forecast_dict=forecast_dict,
#             forecast_name=forecast_name,
#             obs_source=obs_source,
#             region_name=region_name,
#             metric_ = metric_,
#             df_crps_2d = unet_file,
#             better_than=better_than
#         )
        


# Plot either CRPS or CRPS in a map (actual values)
## There are a lot of very negative values for CRPS which skew overall results
## And the tickers in matplotlib are > 30,000

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm
import numpy as np

def save_horizontal_colorbar(cmap, norm, levels, label, out_path):
    fig, ax = plt.subplots(figsize=(10, 1.2))
    fig.subplots_adjust(bottom=0.5, left=0.05, right=0.95)

    cbar = fig.colorbar(
        ScalarMappable(norm=norm, cmap=cmap),
        cax=ax,
        orientation='horizontal',
        boundaries=levels,
        ticks=levels,
        extend='both'
    )

    cbar.set_label(label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=12, length=6)
    cbar.ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))  # <-- Format to 3 decimal places

    plt.savefig(out_path, bbox_inches='tight', dpi=300)
    plt.close()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import numpy as np
from matplotlib.ticker import FormatStrFormatter

def plot_combined_CRPS_map_grid(df_crps_list, df_crps_2d, forecast_name, cmap, obs_source, region_name, metric_):
    '''Plots line plots (row 0) and gridded CRPS maps (rows 1–4) using df_crps_2d'''

    new_dir = f'Outputs/{metric_}_spatial_plots/{region_name}'
    os.makedirs(new_dir, exist_ok=True)
    
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    leads = [1, 2, 3, 4]
    marker_style = ['o', 'v', '^', '<', '>', 's', 'p', '*', '+']
    palette = plt.get_cmap('tab10')

    fig, axs = plt.subplots(5, 4, figsize=(24, 17), dpi=300)

    # Get global min/max for the line plots
    crps_max = max(df[metric_].max() for df in df_crps_list)
    crps_min = min(df[metric_].min() for df in df_crps_list)

    if metric_ == 'CRPSS':
        crps_min += -0.1
        crps_max += +0.3
    else:
        crps_min += -0.003
        crps_max += +0.003
        # crps_min =0.04
        # crps_max =0.0

    # ✅ Inject Cartopy projection only in rows 1–4 (map rows)
    for row in range(1, 5):
        for col in range(4):
            axs[row, col].remove()
            axs[row, col] = fig.add_subplot(5, 4, row * 4 + col + 1, projection=ccrs.PlateCarree())

    # ------------------ Line Plots (Top Row Only) ------------------
    for col_idx, data in enumerate(df_crps_list):
        ax = axs[0, col_idx]
        df = data.copy()  # <-- safe!
        grouped = df.groupby('Forecast')

        for i, (name, group) in enumerate(grouped):
            color = palette(i)
            marker = marker_style[i % len(marker_style)]
            ax.plot(group['Week'], group[metric_], label=name, color=color,
                    marker=marker, linestyle='-', markersize=6)

        ax.set_xlim(0.5, 4.5)
        ax.set_xticks([1, 2, 3, 4])
        ax.xaxis.set_major_locator(ticker.FixedLocator([1, 2, 3, 4]))
        ax.grid(True, linestyle='--', linewidth=0.5)
        ax.tick_params(labelsize=10)
        ax.set_ylim(crps_min, crps_max)
        ax.set_title(seasons[col_idx], fontsize=20)
        ax.set_xlabel('Lead Week', fontsize=12)
        if col_idx == 0:
            ax.set_ylabel(metric_, fontsize=14)
            ax.legend(title='Forecast', fontsize=9, title_fontsize=10)

    # ------------------ Compute global color scale from df_crps_2d ------------------
    vmin_map = np.inf
    vmax_map = -np.inf
    for key in df_crps_2d:
        name= putils.xarray_varname(df_crps_2d[key])
        data = df_crps_2d[key][name].values
        vmin_map = min(vmin_map, np.nanmin(data))
        vmax_map = max(vmax_map, np.nanmax(data))

    vmin_map = np.floor(vmin_map * 10) / 10
    vmax_map = np.ceil(vmax_map * 10) / 10
    cmap = plt.get_cmap(cmap)
    if metric_ =='CRPSS':
        norm = TwoSlopeNorm(vmin=vmin_map, vcenter=0, vmax=vmax_map)
    else:
        norm = Normalize(vmin=0, vmax=0.096)

    if metric_ == 'CRPSS':
        levels = np.arange(vmin_map, vmax_map, 1)
    else:
        levels = np.arange(0, 0.096,0.006)

    # ------------------ CRPS Map Plots (Rows 1–4) ------------------
    for row_idx, lead in enumerate(leads, start=1):
        for col_idx, season in enumerate(seasons):
            key = f'{season}_lead{lead}'
            ax = axs[row_idx, col_idx]
            ax.set_aspect('auto')

            # if key not in df_crps_2d or 'CRPS' not in df_crps_2d[key].data_vars:
            #     ax.set_title(f'{key}\n(No data)', fontsize=10)
            #     ax.coastlines()
            #     continue

            name = putils.xarray_varname(df_crps_2d[key])
            ds = df_crps_2d[key]
            data = ds[name]
            lon = ds.lon.values
            lat = ds.lat.values
            mesh_lon, mesh_lat = np.meshgrid(lon, lat)

            
            im = ax.contourf(mesh_lon, mesh_lat, data.values,
                             levels=levels,
                             transform=ccrs.PlateCarree(), cmap=cmap, norm=norm, extend='both')

            ax.coastlines()
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.set_title(f'Season: {season}, Week: {lead}', fontsize=14)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xformatter = LongitudeFormatter()
            gl.yformatter = LatitudeFormatter()
            gl.xlabel_style = {'size': 8}
            gl.ylabel_style = {'size': 8}

    plt.tight_layout(rect=[0, 0, 0.93, 1])

    # Colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.945, 0.12, 0.020, 0.60])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='both',
                        boundaries=levels, ticks=levels)
    cbar.set_label(f'{metric_} (DL-DM-{forecast_name})', fontsize=14, rotation=90, labelpad=5)
    cbar.ax.tick_params(labelsize=14, pad=2)
    cbar.ax.xaxis.set_major_formatter(FormatStrFormatter('%.3f'))
    
    # Row labels (A–E)
    row_labels = ['A', 'B', 'C', 'D', 'E']
    for row_idx, label in enumerate(row_labels):
        axs[row_idx, 0].text(
            -0.09, 1.05,
            label,
            transform=axs[row_idx, 0].transAxes,
            fontsize=20, fontweight='bold', va='top', ha='right'
        )

    # Example usage
    save_horizontal_colorbar(
        cmap=cmap,
        norm=norm,
        levels=levels,
        label=f'{metric_}',
        out_path=f'{new_dir}/colorbar_{metric_}_{forecast_name}_{obs_source}.png'
    )
    plt.savefig(f'{new_dir}/Combined_5x4_{metric_}_only_{forecast_name}_{obs_source}_NO_BIAS_correction.png', dpi=300)
    plt.show()


In [ ]:
for metric_ in ['CRPS','CRPS']:
    for forecast_name in ['ECMWF','GEFSv12']:
        if forecast_name == 'GEFSv12' and metric_=='CRPS':
            unet_file = unet_crps_2d_gefsv12
            df_crps_list=[df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son]
        elif forecast_name == 'GEFSv12' and metric_=='CRPS':
            unet_file = unet_CRPS_2d_gefsv12
            df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
        elif forecast_name == 'ECMWF' and metric_=='CRPS':
            unet_file = unet_CRPS_2d_ecmwf
            df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
        elif forecast_name == 'ECMWF' and metric_=='CRPS':
            unet_file = unet_crps_2d_ecmwf
            df_crps_list=[df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son]            

    
        plot_combined_CRPS_map_grid(
            df_crps_list=df_crps_list,
            forecast_name=forecast_name,
            cmap='bwr' if metric_ == 'CRPSS' else 'YlOrRd_r',
            obs_source=obs_source,
            region_name=region_name,
            metric_ = metric_,
            df_crps_2d = unet_file,
        )
        
stop

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import numpy as np

def plot_combined_CRPS_map_grid_by_lead(df_crps_list, df_crps_2d, forecast_name, cmap, obs_source, region_name, metric_,week_leads):
    '''Plots line plots (row 0) and gridded CRPS maps (rows 1–4) using df_crps_2d'''

    new_dir = f'Outputs/{metric_}_spatial_plots/{region_name}'
    os.makedirs(new_dir, exist_ok=True)
    
    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    leads = [1, 2, 3, 4]
    marker_style = ['o', 'v', '^', '<', '>', 's', 'p', '*', '+']
    palette = plt.get_cmap('tab10')

    fig, axs = plt.subplots(1 + (len(week_leads)), 4, figsize=(17, 10), dpi=300)

    # Get global min/max for the line plots
    crps_max = max(df[metric_].max() for df in df_crps_list)
    crps_min = min(df[metric_].min() for df in df_crps_list)

    if metric_ == 'CRPSS':
        crps_min += -0.1
        crps_max += +0.3
    else:
        crps_min =0.04
        crps_max = -0.02

    # ✅ Inject Cartopy projection only in rows 1–4 (map rows)
    for row in range(1, 2):
        for col in range(4):
            axs[row, col].remove()
            axs[row, col] = fig.add_subplot(1 + (len(week_leads)), 4, row * 4 + col + 1, projection=ccrs.PlateCarree())

    # ------------------ Line Plots (Top Row Only) ------------------
    for col_idx, data in enumerate(df_crps_list):
        ax = axs[0, col_idx]
        df = data.copy()  # <-- safe!
        grouped = df.groupby('Forecast')

        for i, (name, group) in enumerate(grouped):
            color = palette(i)
            marker = marker_style[i % len(marker_style)]
            ax.plot(group['Week'], group[metric_], label=name, color=color,
                    marker=marker, linestyle='-', markersize=6)

        ax.set_xlim(0.5, 4.5)
        ax.set_xticks([1, 2, 3, 4])
        ax.xaxis.set_major_locator(ticker.FixedLocator([1, 2, 3, 4]))
        ax.grid(True, linestyle='--', linewidth=0.5)
        ax.tick_params(labelsize=10)
        ax.set_ylim(crps_min, crps_max)
        ax.set_title(seasons[col_idx], fontsize=20)
        ax.set_xlabel('Lead Week', fontsize=12)
        if col_idx == 0:
            ax.set_ylabel(metric_, fontsize=14)
            ax.legend(title='Forecast', fontsize=9, title_fontsize=10)

    # ------------------ Compute global color scale from df_crps_2d ------------------
    vmin_map = np.inf
    vmax_map = -np.inf
    for key in df_crps_2d:
        name= putils.xarray_varname(df_crps_2d[key])
        data = df_crps_2d[key][name].values
        vmin_map = min(vmin_map, np.nanmin(data))
        vmax_map = max(vmax_map, np.nanmax(data))

    vmin_map = np.floor(vmin_map * 10) / 10
    vmax_map = np.ceil(vmax_map * 10) / 10
    cmap = plt.get_cmap(cmap)
    if metric_ =='CRPSS':
        norm = TwoSlopeNorm(vmin=vmin_map, vcenter=0, vmax=vmax_map)
    else:
        norm = Normalize(vmin=vmin_map, vmax=vmax_map)

    if metric_ == 'CRPSS':
        levels = np.arange(vmin_map, vmax_map, 1)
    else:
        levels = np.arange(vmin_map, vmax_map,0.006)

    # ------------------ CRPS Map Plots (Rows 1–4) ------------------
    for row_idx, lead in enumerate(week_leads, start=1):
        for col_idx, season in enumerate(seasons):
            key = f'{season}_lead{lead}'
            ax = axs[row_idx, col_idx]
            ax.set_aspect('auto')

            # if key not in df_crps_2d or 'CRPS' not in df_crps_2d[key].data_vars:
            #     ax.set_title(f'{key}\n(No data)', fontsize=10)
            #     ax.coastlines()
            #     continue

            name = putils.xarray_varname(df_crps_2d[key])
            ds = df_crps_2d[key]
            data = ds[name]
            lon = ds.lon.values
            lat = ds.lat.values
            mesh_lon, mesh_lat = np.meshgrid(lon, lat)

            
            im = ax.contourf(mesh_lon, mesh_lat, data.values,
                             levels=levels,
                             transform=ccrs.PlateCarree(), cmap=cmap, norm=norm, extend='both')

            ax.coastlines()
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.set_title(f'Season: {season}, Week: {lead}', fontsize=14)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xformatter = LongitudeFormatter()
            gl.yformatter = LatitudeFormatter()
            gl.xlabel_style = {'size': 8}
            gl.ylabel_style = {'size': 8}

    plt.tight_layout(rect=[0, 0, 0.93, 1])

    # Colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.945, 0.12, 0.020, 0.60])
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='both',
                        boundaries=levels, ticks=levels)
    cbar.set_label(f'{metric_} (DL-DM-{forecast_name})', fontsize=14, rotation=90, labelpad=5)
    cbar.ax.tick_params(labelsize=14, pad=2)

    plt.savefig(f'{new_dir}/Combined_Wk{week_leads[0]}_5x4_{metric_}_only_{forecast_name}_{obs_source}_NO_BIAS_correction.png', dpi=300)
    plt.show()


In [ ]:
for metric_ in ['CRPS','CRPS']:
    for forecast_name in ['ECMWF','GEFSv12']:
        if forecast_name == 'GEFSv12' and metric_=='CRPS':
            unet_file = unet_crps_2d_gefsv12.copy()
            df_crps_list=[df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son]
        elif forecast_name == 'GEFSv12' and metric_=='CRPS':
            unet_file = unet_CRPS_2d_gefsv12.copy()
            df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
        elif forecast_name == 'ECMWF' and metric_=='CRPS':
            unet_file = unet_CRPS_2d_ecmwf.copy()
            df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
        elif forecast_name == 'ECMWF' and metric_=='CRPS':
            unet_file = unet_crps_2d_ecmwf.copy()
            df_crps_list=[df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son]            

    
        plot_combined_CRPS_map_grid_by_lead(
            df_crps_list=df_crps_list,
            forecast_name=forecast_name,
            cmap='bwr' if metric_ == 'CRPSS' else 'YlOrRd_r',
            obs_source=obs_source,
            region_name=region_name,
            metric_ = metric_,
            df_crps_2d = unet_file,
            week_leads=[3]
        )
        


In [ ]:
stop

In [ ]:
#Still working on the code below if I need it

# CRPS and CRPS improvement

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import numpy as np

def improvement_combined_CRPS_map_grid(df_crps_list, df_crps_2d, forecast_name, cmap, obs_source, region_name, metric_, forecast_dict):
    '''Plots line plots (row 0) and gridded CRPS or CRPS diff maps (rows 1–4) using df_crps_2d'''

    seasons = ['DJF', 'MAM', 'JJA', 'SON']
    leads = [1, 2, 3, 4]
    marker_style = ['o', 'v', '^', '<', '>', 's', 'p', '*', '+']
    palette = plt.get_cmap('tab10')

    fig, axs = plt.subplots(5, 4, figsize=(24, 17), dpi=300)

    # Line plot y-axis limits
    crps_max = max(df[metric_].max() for df in df_crps_list)
    crps_min = min(df[metric_].min() for df in df_crps_list)

    if metric_ == 'CRPS':
        crps_min += -0.1
        crps_max += +0.3
    else:
        crps_min += -0.003
        crps_max += +0.003

    # Inject Cartopy projections for map rows (1–4)
    for row in range(1, 5):
        for col in range(4):
            axs[row, col].remove()
            axs[row, col] = fig.add_subplot(5, 4, row * 4 + col + 1, projection=ccrs.PlateCarree())

    # ------------------ Line Plots (Top Row) ------------------
    for col_idx, df in enumerate(df_crps_list):
        ax = axs[0, col_idx]
        grouped = df.groupby('Forecast')

        for i, (name, group) in enumerate(grouped):
            color = palette(i)
            marker = marker_style[i % len(marker_style)]
            ax.plot(group['Week'], group[metric_], label=name, color=color,
                    marker=marker, linestyle='-', markersize=6)

        ax.set_xlim(0.5, 4.5)
        ax.set_xticks([1, 2, 3, 4])
        ax.xaxis.set_major_locator(ticker.FixedLocator([1, 2, 3, 4]))
        ax.grid(True, linestyle='--', linewidth=0.5)
        ax.tick_params(labelsize=10)
        ax.set_ylim(crps_min, crps_max)
        ax.set_title(seasons[col_idx], fontsize=20)
        ax.set_xlabel('Lead Week', fontsize=12)
        if col_idx == 0:
            ax.set_ylabel(metric_, fontsize=14)
            ax.legend(title='Forecast', fontsize=9, title_fontsize=10)

    # ------------------ Global Color Scale ------------------
    vmin_map = np.inf
    vmax_map = -np.inf
    for key in df_crps_2d:
        name = putils.xarray_varname(df_crps_2d[key])
        season, lead_str = key.split('_')
        lead_idx = int(lead_str.replace('lead', '')) - 1  # 0-based index
        unet_data = df_crps_2d[key][name].values
        forecast_data = forecast_dict[season][name].isel(lead=lead_idx).values

        if metric_ == 'CRPS':
            diff = forecast_data - unet_data
        else:  # CRPS
            diff = unet_data - forecast_data
            
        print(np.nanmin(diff))
        print(np.nanmax(diff))

        vmin_map = min(vmin_map, np.nanmin(diff))
        vmax_map = max(vmax_map, np.nanmax(diff))

    vmin_map = np.floor(vmin_map * 10) / 10
    vmax_map = np.ceil(vmax_map * 10) / 10

    # Use red → yellow colormap, with appropriate normalization
    cmap = plt.get_cmap('YlOrRd')
    norm = TwoSlopeNorm(vmin=vmin_map, vcenter=0, vmax=vmax_map)
    levels = np.linspace(vmin_map, vmax_map, num=20)

    # ------------------ Map Plots (Rows 1–4) ------------------
    for row_idx, lead in enumerate(leads, start=1):
        for col_idx, season in enumerate(seasons):
            key = f'{season}_lead{lead}'
            ax = axs[row_idx, col_idx]
            ax.set_aspect('auto')


            name = putils.xarray_varname(df_crps_2d[key])
            ds = df_crps_2d[key]
            unet_data = ds[name].values
            forecast_data = forecast_dict[season][name].isel(lead=lead - 1).values

            if metric_ == 'CRPS':
                plot_data = forecast_data - unet_data
            else:  # CRPS
                plot_data = unet_data - forecast_data

            lon = ds.lon.values
            lat = ds.lat.values
            mesh_lon, mesh_lat = np.meshgrid(lon, lat)

            im = ax.contourf(mesh_lon, mesh_lat, plot_data, levels=levels,
                             transform=ccrs.PlateCarree(), cmap=cmap, norm=norm, extend='both')

            ax.coastlines()
            ax.add_feature(cfeature.BORDERS, linewidth=0.5)
            ax.set_title(f'Season: {season}, Week: {lead}', fontsize=14)

            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xformatter = LongitudeFormatter()
            gl.yformatter = LatitudeFormatter()
            gl.xlabel_style = {'size': 8}
            gl.ylabel_style = {'size': 8}

    plt.tight_layout(rect=[0, 0, 0.93, 1])

    # Colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.945, 0.12, 0.020, 0.60])
    label_map = {
        'CRPS': f'Diff = {forecast_name} - UNET',
        'CRPS': f'Diff = UNET - {forecast_name}'
    }
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical', extend='both',
                        boundaries=levels, ticks=levels)
    cbar.set_label(label_map[metric_], fontsize=14, rotation=90, labelpad=5)

    cbar.ax.tick_params(labelsize=14, pad=2)

    # Row labels (A–E)
    row_labels = ['A', 'B', 'C', 'D', 'E']
    for row_idx, label in enumerate(row_labels):
        axs[row_idx, 0].text(
            -0.09, 1.05,
            label,
            transform=axs[row_idx, 0].transAxes,
            fontsize=20, fontweight='bold', va='top', ha='right'
        )

    plt.savefig(f'Outputs/ACC_CRPS_line_plots/{region_name}/IMRPROVMENT_5x4_{metric_}_only_{forecast_name}_{obs_source}_Map.png', dpi=300)
    plt.show()


In [ ]:
for metric_ in ['CRPS','CRPS']:
    for forecast_name in ['ECMWF','GEFSv12']:
        if forecast_name == 'GEFSv12' and metric_=='CRPS':
            forecast_dict = gefsv12_bc_crps_2d
            df_crps_2d = unet_crps_2d
            df_crps_list=[df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son]
        elif forecast_name == 'GEFSv12' and metric_=='CRPS':
            forecast_dict = gefsv12_bc_CRPS_2d
            df_crps_2d = unet_CRPS_2d
            df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
        elif forecast_name == 'ECMWF' and metric_=='CRPS':
            forecast_dict = ecmwf_bc_CRPS_2d
            df_crps_2d = unet_CRPS_2d
            df_crps_list=[df_CRPS_djf, df_CRPS_mam, df_CRPS_jja, df_CRPS_son]
        elif forecast_name == 'ECMWF' and metric_=='CRPS':
            forecast_dict = ecmwf_bc_CRPS_2d
            df_crps_2d = unet_crps_2d
            df_crps_list=[df_crps_djf, df_crps_mam, df_crps_jja, df_crps_son]            

    
        improvement_combined_CRPS_map_grid(
            df_crps_list=df_crps_list,
            forecast_dict=forecast_dict,
            forecast_name=forecast_name,
            cmap='bwr' if metric_ == 'CRPS' else 'YlOrRd_r',
            obs_source=obs_source,
            region_name=region_name,
            metric_ = metric_,
            df_crps_2d = df_crps_2d,
        )
        
